# F1: スライド用図の再生成（統一カラー）

**凡例ルール（全図共通）**
- 赤 `#E8534A` = 上昇・プラス
- 青 `#3A7FC1` = 下落・マイナス
- 灰 `#CCCCCC` = 据え置き・その他

このノートブックを順番に実行すると、スライド `PBL発表_v2.pptx` で使用する全図が `/outputs/` に保存されます。

In [ ]:
import sys
sys.path.insert(0, "../src")

from pathlib import Path
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.lines as mlines
import matplotlib.patches as mpatches
from matplotlib.colors import TwoSlopeNorm

try:
    import japanize_matplotlib
except ImportError:
    pass

from compare_years import build_panel, donut_effect_panel

plt.rcParams["figure.dpi"] = 120

PATH_2022 = Path("/Users/KASU/REX_data_2020-2022/2022/nouhin_line_2022.shp")
PATH_2020 = Path("/Users/KASU/REX_data_2020-2022/2020/nouhin_line_2020.shp")
OUT_DIR   = Path("../outputs")
OUT_DIR.mkdir(exist_ok=True)

# 統一カラー
RED  = "#E8534A"   # 上昇
BLUE = "#3A7FC1"   # 下落
GRAY = "#CCCCCC"   # 据え置き
NAVY = "#1B2B4B"
GOLD = "#D4A843"

print("セットアップ完了")

In [ ]:

# パネルデータ読み込み（E1と同じ処理）
panel = build_panel(PATH_2022, PATH_2020)

# 路線タイプ分類（5タイプ）
def classify_road(row):
    pre, covid, rec = row["chg_18_19"], row["chg_20_21"], row["chg_21_22"]
    if pd.isna(pre) or pd.isna(covid) or pd.isna(rec):
        return "不明"
    if pre < 0:
        return "構造的下落"
    if pre > 0 and covid > 0 and rec > 0:
        return "構造的上昇"
    if covid < 0:
        return "回復型" if rec > 0 else "コロナ型下落"
    return "安定"

panel["road_type"] = panel.apply(classify_road, axis=1)
counts = panel["road_type"].value_counts()
total = counts.sum()
print("路線タイプ分布:")
for t in ["構造的上昇", "安定", "回復型", "コロナ型下落", "構造的下落", "不明"]:
    n = counts.get(t, 0)
    print(f"  {t}: {n:,}件 ({n/total*100:.1f}%)")
print(f"パネル件数: {len(panel):,}")


---
## 図1: 全国路線価 年次変化率の推移
スライド4で使用

In [ ]:

# 目標AR: 6.1 / 3.1 = 1.97 → figsize=(10, 5.08)
years   = ["2018→2019", "2019→2020", "2020→2021", "2021→2022"]
cols    = ["chg_18_19", "chg_19_20", "chg_20_21", "chg_21_22"]
means   = [panel[c].mean() for c in cols]

fig, ax = plt.subplots(figsize=(10, 5.08))

bar_colors = [RED if v >= 0 else BLUE for v in means]
ax.bar(range(len(years)), means, color=bar_colors, alpha=0.75, width=0.5)
ax.plot(range(len(years)), means, marker="o", color=NAVY, linewidth=2, zorder=5, label="平均")

ax.axhline(0, color="black", linewidth=0.8, linestyle="--")
ax.axvspan(1.5, 3.5, alpha=0.06, color="orange")
ax.set_xticks(range(len(years)))
ax.set_xticklabels(years, fontsize=11)
ax.set_ylabel("路線価変化率 (%)", fontsize=11)
ax.set_title("全国路線価 年次変化率の推移（2018〜2022）", fontsize=13)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"{v:+.2f}%"))

leg = [
    mpatches.Patch(color=RED,  label="平均がプラス（上昇）"),
    mpatches.Patch(color=BLUE, label="平均がマイナス（下落）"),
    mpatches.Patch(color="orange", alpha=0.3, label="コロナ禍（2020〜2022）"),
    mlines.Line2D([], [], color=NAVY, marker="o", label="全国平均"),
]
ax.legend(handles=leg, fontsize=9)

plt.tight_layout()
plt.savefig(OUT_DIR / "corona_yearly_change.png", dpi=150, bbox_inches="tight")
plt.show()
print("保存: corona_yearly_change.png")


---
## 図2: 路線レベル 変動傾向マップ（4期間）
スライド5で使用

In [ ]:

# 4期間を1枚ずつ個別PNGとして保存
# 2×2グリッドスライドに収めるため横長AR=2.0 → figsize=(10, 5)
panel_pt = panel.copy()
panel_pt["geometry"] = panel.geometry.centroid
panel_pt = panel_pt.to_crs(epsg=3857)

periods = [
    ("① コロナ前\n2018→2019",    "chg_18_19", "corona_map_road_01.png"),
    ("② コロナ前\n2019→2020",    "chg_19_20", "corona_map_road_02.png"),
    ("③ コロナ禍\n2020→2021",    "chg_20_21", "corona_map_road_03.png"),
    ("④ 回復期\n2021→2022",      "chg_21_22", "corona_map_road_04.png"),
]
SAMPLE_FLAT = 80_000
SEED = 42

for label, col, fname in periods:
    sub = panel_pt[[col, "geometry"]].dropna(subset=[col])
    up   = sub[sub[col] > 0]
    flat = sub[sub[col] == 0]
    dn   = sub[sub[col] < 0]
    if len(flat) > SAMPLE_FLAT:
        flat = flat.sample(SAMPLE_FLAT, random_state=SEED)

    fig, ax = plt.subplots(figsize=(10, 5))
    fig.patch.set_facecolor("#1a1a2e")
    ax.set_facecolor("#1a1a2e")

    flat.plot(ax=ax, color=GRAY, markersize=0.3, alpha=0.25)
    up.plot(  ax=ax, color=RED,  markersize=0.6, alpha=0.5)
    dn.plot(  ax=ax, color=BLUE, markersize=0.9, alpha=0.7)

    try:
        import contextily as ctx
        ctx.add_basemap(ax, crs=panel_pt.crs.to_string(),
                        source=ctx.providers.CartoDB.DarkMatter, zoom=5, alpha=0.45)
    except Exception:
        pass

    leg = [
        mlines.Line2D([], [], color=RED,  marker="o", ls="none", markersize=7,
                      label=f"上昇  {len(up):,}件"),
        mlines.Line2D([], [], color=GRAY, marker="o", ls="none", markersize=7,
                      label=f"据え置き {len(sub[sub[col]==0]):,}件"),
        mlines.Line2D([], [], color=BLUE, marker="o", ls="none", markersize=7,
                      label=f"下落  {len(dn):,}件"),
    ]
    ax.legend(handles=leg, loc="lower left", fontsize=10,
              frameon=True, framealpha=0.85, facecolor="#222233", labelcolor="white")
    ax.set_title(label, fontsize=16, fontweight="bold", color="white", pad=12)
    ax.set_axis_off()

    plt.tight_layout()
    plt.savefig(OUT_DIR / fname, dpi=150, bbox_inches="tight",
                facecolor=fig.get_facecolor())
    plt.show()
    print(f"保存: {fname}")


---
## 図3: Donut Effect（東京）
スライド6で使用

In [ ]:

# 目標AR: 6.8 / 3.2 = 2.13 → figsize=(13, 6.10)
# ラベル修正: 2019→2020は路線価評価が1月1日基準のためコロナ影響なし→「前」
donut = donut_effect_panel(panel, city="東京")

fig, axes = plt.subplots(1, 2, figsize=(13, 6.10))
bands = donut.index.astype(str)
x = range(len(bands))

# 左: 距離帯別・年次変化率
# 期間ごとに明確に異なる色を使う（上昇/下落の赤青とは別軸）
ax = axes[0]
period_styles = [
    ("pre_median",   "2018→2019（コロナ前）",  "#888888", "-",  "o"),
    ("p1920_median", "2019→2020（コロナ前）",  "#5BAD6F", "--", "s"),  # 緑=前年同様
    ("p2021_median", "2020→2021（コロナ禍）",  BLUE,      "-",  "^"),  # 青=下落期
    ("p2122_median", "2021→2022（回復期）",    RED,       "-",  "D"),  # 赤=上昇期
]
for col, label, color, ls, marker in period_styles:
    ax.plot(x, donut[col], marker=marker, label=label,
            color=color, linewidth=2, linestyle=ls, markersize=6)
ax.axhline(0, color="black", linewidth=0.7, linestyle="--")
ax.set_xticks(x)
ax.set_xticklabels(bands, rotation=30, ha="right", fontsize=9)
ax.set_ylabel("変化率中央値 (%)", fontsize=10)
ax.set_title("東京からの距離帯別 年次変化率", fontsize=11)
ax.legend(fontsize=9)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"{v:+.2f}%"))

# 右: Donut Diff ── 赤=コロナ前より上昇 / 青=コロナ前より下落
ax = axes[1]
diff = donut["donut_diff"]
bar_colors = [RED if v >= 0 else BLUE for v in diff]
ax.bar(x, diff, color=bar_colors, alpha=0.85)
ax.axhline(0, color="black", linewidth=0.8, linestyle="--")
ax.set_xticks(x)
ax.set_xticklabels(bands, rotation=30, ha="right", fontsize=9)
ax.set_ylabel("コロナ禍累計 − コロナ前 (pp)", fontsize=10)
ax.set_title("Donut Effect: コロナ禍の上乗せ変化率\n（赤=コロナ前より上昇 / 青=コロナ前より下落）", fontsize=11)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"{v:+.2f}"))

plt.suptitle("Donut Effect 検証（東京）", fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(OUT_DIR / "corona_donut_effect.png", dpi=150, bbox_inches="tight")
plt.show()
print("保存: corona_donut_effect.png")


---
## 図4: 路線タイプ分類マップ（路線レベル直接描画）
スライド8で使用

**計算方法（旧・都道府県集計）の問題点:**
- 各路線を4タイプ（構造的下落/コロナ型下落/回復型/安定・上昇）に分類
- 都道府県ごとに「コロナ型下落%」−「構造的下落%」を計算
- → 都道府県内のばらつきが消える・同一都道府県の都市部/農村部の差が見えない

**新しいアプローチ（路線レベル直接）:**
- 各路線の重心点を4タイプの色で直接プロット
- 空間パターンが都道府県の行政境界に縛られず見える

In [ ]:

# 目標AR: 7.6 / 4.4 = 1.73 → figsize=(11, 6.36)
# 路線レベルで5タイプを直接描画

TYPE_COLORS = {
    "安定":       ("#444455", 0.12, 0.3),   # 背景として暗く薄く
    "構造的上昇": (RED,       0.7,  1.0),   # 赤（構造的に強い）
    "回復型":     ("#5BAD6F", 0.7,  0.8),   # 緑
    "コロナ型下落": ("#E8A838", 0.7, 0.9),  # 橙
    "構造的下落": (BLUE,      0.8,  1.0),   # 青（構造的に弱い）
}
SAMPLE_BG = 100_000   # 安定は間引き

panel_type_pt = panel[panel["road_type"].isin(TYPE_COLORS)].copy()
panel_type_pt["geometry"] = panel_type_pt.geometry.centroid
panel_type_pt = panel_type_pt.to_crs(epsg=3857)

fig, ax = plt.subplots(figsize=(11, 6.36))
fig.patch.set_facecolor("#1a1a2e")
ax.set_facecolor("#1a1a2e")

# 描画順: 安定(背景) → 回復型 → コロナ型 → 構造的下落 → 構造的上昇(前面)
draw_order = ["安定", "回復型", "コロナ型下落", "構造的下落", "構造的上昇"]
for t in draw_order:
    color, alpha, _ = TYPE_COLORS[t]
    sub = panel_type_pt[panel_type_pt["road_type"] == t]
    if t == "安定" and len(sub) > SAMPLE_BG:
        sub = sub.sample(SAMPLE_BG, random_state=42)
    sub.plot(ax=ax, color=color, markersize=0.5, alpha=alpha)

try:
    import contextily as ctx
    ctx.add_basemap(ax, crs=panel_type_pt.crs.to_string(),
                    source=ctx.providers.CartoDB.DarkMatter, zoom=5, alpha=0.4)
except Exception:
    pass

counts = panel["road_type"].value_counts()
leg = [
    mlines.Line2D([], [], color=RED,      marker="o", ls="none", markersize=7,
                  label=f"構造的上昇  {counts.get('構造的上昇',0):,}件"),
    mlines.Line2D([], [], color="#5BAD6F", marker="o", ls="none", markersize=7,
                  label=f"回復型  {counts.get('回復型',0):,}件"),
    mlines.Line2D([], [], color="#E8A838", marker="o", ls="none", markersize=7,
                  label=f"コロナ型下落  {counts.get('コロナ型下落',0):,}件"),
    mlines.Line2D([], [], color=BLUE,      marker="o", ls="none", markersize=7,
                  label=f"構造的下落  {counts.get('構造的下落',0):,}件"),
    mlines.Line2D([], [], color="#444455", marker="o", ls="none", markersize=5,
                  label=f"安定  {counts.get('安定',0):,}件"),
]
ax.legend(handles=leg, loc="lower left", fontsize=9,
          frameon=True, framealpha=0.85, facecolor="#222233", labelcolor="white")

ax.set_axis_off()
ax.set_title("路線タイプ分類マップ（5タイプ）\n赤=構造的上昇 / 青=構造的下落 / 橙=コロナ型下落 / 緑=回復型",
             fontsize=12, fontweight="bold", color="white", pad=10)

plt.tight_layout()
plt.savefig(OUT_DIR / "corona_road_type_map.png", dpi=150, bbox_inches="tight",
            facecolor=fig.get_facecolor())
plt.show()
print("保存: corona_road_type_map.png")


---
## 図5: ずっと上昇・ずっと下落マップ
スライド9で使用

In [ ]:
chg_cols = ["chg_18_19", "chg_19_20", "chg_20_21", "chg_21_22"]
mask_valid  = panel[chg_cols].notna().all(axis=1)
p = panel[mask_valid].copy()
always_up   = p[(p[chg_cols] > 0).all(axis=1)]
always_down = p[(p[chg_cols] < 0).all(axis=1)]

def to_pt(gdf):
    g = gdf.copy()
    g["geometry"] = gdf.geometry.centroid
    return g.to_crs(epsg=3857)

pt_up   = to_pt(always_up)
pt_down = to_pt(always_down)
other_idx = p.index.difference(always_up.index).difference(always_down.index)
other_sample = to_pt(p.loc[other_idx].sample(min(100_000, len(other_idx)), random_state=42))

fig, ax = plt.subplots(figsize=(12, 10))
fig.patch.set_facecolor("#1a1a2e")
ax.set_facecolor("#1a1a2e")

other_sample.plot(ax=ax, color="#444466", markersize=0.2, alpha=0.2)
pt_down.plot(    ax=ax, color=BLUE,      markersize=1.2, alpha=0.6)
pt_up.plot(      ax=ax, color=RED,       markersize=1.5, alpha=0.7)

try:
    import contextily as ctx
    ctx.add_basemap(ax, crs=pt_up.crs.to_string(),
                    source=ctx.providers.CartoDB.DarkMatter, zoom=5, alpha=0.4)
except Exception:
    pass

leg = [
    mlines.Line2D([], [], color=RED,      marker="o", ls="none", markersize=7,
                  label=f"ずっと上昇（全4期間）  {len(pt_up):,}件"),
    mlines.Line2D([], [], color=BLUE,     marker="o", ls="none", markersize=7,
                  label=f"ずっと下落（全4期間）  {len(pt_down):,}件"),
    mlines.Line2D([], [], color="#444466",marker="o", ls="none", markersize=5,
                  label="その他"),
]
ax.legend(handles=leg, loc="lower left", fontsize=10,
          frameon=True, framealpha=0.85, facecolor="#222233", labelcolor="white")
ax.set_axis_off()
ax.set_title("コロナと無関係に『ずっと上昇』『ずっと下落』だった路線（2018〜2022 全4期間）",
             fontsize=13, fontweight="bold", color="white", pad=12)
plt.tight_layout()
plt.savefig(OUT_DIR / "corona_always_trend.png", dpi=150, bbox_inches="tight",
            facecolor=fig.get_facecolor())
plt.show()
print("保存: corona_always_trend.png")

---
## 全図確認（スライド順）

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

slide_figs = [
    ("スライド3: データ概要",            "map_price.png"),
    ("スライド4: 年次変化率",             "corona_yearly_change.png"),
    ("スライド5: 路線レベルマップ",       "corona_map_road.png"),
    ("スライド6: Donut Effect",          "corona_donut_effect.png"),
    ("スライド8: 下落の地域差",          "corona_road_type_map.png"),
    ("スライド9: ずっと上昇/下落",       "corona_always_trend.png"),
]

for title, fname in slide_figs:
    path = OUT_DIR / fname
    if not path.exists():
        print(f"ファイルなし: {fname}")
        continue
    img = Image.open(path)
    fig, ax = plt.subplots(figsize=(12, 5))
    ax.imshow(img)
    ax.set_axis_off()
    ax.set_title(title, fontsize=13, fontweight="bold", pad=8)
    plt.tight_layout()
    plt.show()

print("\n=== 全図確認完了 ===")